# Module 1 — Document Ingest with `ai_parse_document` + `ai_query`

**Time**: ~30 minutes

**For analysts**: We have a folder of CISA advisory PDFs. We want a structured table of CVEs, affected products, and recommended mitigations — without hand-typing them. Databricks AI Functions do this in two SQL calls.

**For engineers**: `ai_parse_document` runs an OCR / layout-aware document model over a PDF and returns text + structure. `ai_query` lets us call any model serving endpoint inline; with a JSON schema it returns parsed structured output. Both run on the SQL warehouse — no Python, no orchestration framework.

**Goal of this module**: starting from raw PDFs in a UC volume, end with a structured Delta table you can join against `cves` and `kev_catalog`.

```
raw PDFs in volume  ->  ai_parse_document  ->  parsed_advisories
                                              |
                                              ai_query (structured output)
                                              |
                                              v
                                       structured_advisories
```

## 1. List the raw advisory PDFs

In [ ]:
%sql
LIST '/Volumes/disa_workshop/threat_intel/raw_advisories'

## 2. Parse the PDFs with `ai_parse_document`

Each row in `parsed_advisories` is one PDF + its extracted text content. Runs in parallel over the warehouse — should finish in under a minute for 20 PDFs.

In [ ]:
%sql
CREATE OR REPLACE TABLE disa_workshop.threat_intel.parsed_advisories AS
SELECT
  path,
  ai_parse_document(content) AS parsed
FROM READ_FILES(
  '/Volumes/disa_workshop/threat_intel/raw_advisories',
  format => 'binaryFile'
);

SELECT path, parsed.document.pages[0].content AS first_page_preview
FROM disa_workshop.threat_intel.parsed_advisories
LIMIT 3;

## 3. Pull structured threat intel with `ai_query`

We use a structured output schema to force the model to return clean JSON: a list of CVEs, affected products, and recommended actions per advisory.

**Why this matters for DISA**: this is what a CSSP analyst does manually today — open a PDF, copy CVE IDs into a spreadsheet, link them to the asset inventory. We're collapsing that into a SQL query.

In [ ]:
%sql
CREATE OR REPLACE TABLE disa_workshop.threat_intel.structured_advisories AS
SELECT
  path,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Extract structured threat intelligence from this CISA advisory. ',
      'Return: title, summary, CVE list, affected vendors and products, ',
      'recommended mitigations, and earliest known exploitation date. ',
      'Advisory text:\n\n',
      LEFT(parsed.document.pages[0].content, 8000)
    ),
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "advisory_extract",
        "schema": {
          "type": "object",
          "properties": {
            "title": {"type": "string"},
            "summary": {"type": "string"},
            "cves": {"type": "array", "items": {"type": "string"}},
            "vendors": {"type": "array", "items": {"type": "string"}},
            "products": {"type": "array", "items": {"type": "string"}},
            "mitigations": {"type": "array", "items": {"type": "string"}},
            "first_exploited": {"type": "string"}
          },
          "required": ["title", "summary", "cves"]
        }
      }
    }'
  ) AS extract
FROM disa_workshop.threat_intel.parsed_advisories;

## 4. Flatten into queryable columns

In [ ]:
%sql
CREATE OR REPLACE VIEW disa_workshop.threat_intel.advisories AS
SELECT
  path,
  parse_json(extract):title::string AS title,
  parse_json(extract):summary::string AS summary,
  from_json(parse_json(extract):cves::string, 'ARRAY<STRING>') AS cves,
  from_json(parse_json(extract):vendors::string, 'ARRAY<STRING>') AS vendors,
  from_json(parse_json(extract):products::string, 'ARRAY<STRING>') AS products,
  from_json(parse_json(extract):mitigations::string, 'ARRAY<STRING>') AS mitigations,
  parse_json(extract):first_exploited::string AS first_exploited
FROM disa_workshop.threat_intel.structured_advisories;

SELECT title, cves, vendors FROM disa_workshop.threat_intel.advisories LIMIT 10;

## 5. Join the structured advisories to KEV

Now we can answer a real DISA question: **"Which advisories cite a CVE that's already in the KEV catalog?"**

In [ ]:
%sql
SELECT
  a.title,
  exploded.cve,
  k.dateAdded AS kev_date_added,
  k.dueDate AS kev_due_date,
  k.requiredAction AS kev_required_action
FROM disa_workshop.threat_intel.advisories a
LATERAL VIEW EXPLODE(a.cves) exploded AS cve
INNER JOIN disa_workshop.threat_intel.kev_catalog k ON exploded.cve = k.cveID
ORDER BY k.dateAdded DESC;

## Try it yourself

1. Modify the schema in step 3 to also extract a `severity_assessment` field. Re-run.
2. Try a different model — swap `databricks-meta-llama-3-3-70b-instruct` for `databricks-claude-sonnet-4` and compare extraction quality.
3. Use `ai_summarize` to produce a 2-sentence daily-brief summary across all advisories: `SELECT ai_summarize(CONCAT_WS('\n', collect_list(summary))) FROM advisories;`

**Next**: open the AI Playground and we'll walk through it together (Module 2). Then **Module 3 — `notebooks/03_genie_setup.ipynb`**.